# EVIDENCIA DE APRENDIZAJE 1: ANÁLISIS Y HERRAMIENTAS DE EXTRACCIÓN DE DATOS
### Unidad 1: Metodologías y Técnicas de Web Scraping
**Institución Universitaria Digital de Antioquia (IU Digital)**  
**Facultad de Ingeniería - Programa de Ingeniería de Datos / Software**  
**Docente:** Tutor del Curso  
**Estudiante:** [Apellido Nombre]  
**Fecha:** Octubre 2026  
**Caso Práctico de Estudio:** Monitoreo y Extracción de Datos del **iPhone 17 Pro Max** en **Mercado Libre Colombia**


## 1. Introducción y Contexto del Problema

El comercio electrónico contemporáneo genera terabytes de datos diarios sobre precios, existencias, promociones y comportamiento del consumidor. En este ecosistema, la extracción automatizada de datos (*Web Scraping*) se consolida como una metodología neurálgica para la **Inteligencia de Precios (*Price Intelligence*)** y la toma de decisiones estratégicas en tiempo real.

En esta primera práctica articulada de la Unidad 1, se analiza la plataforma de marketplace **Mercado Libre Colombia** (`mercadolibre.com.co`), enfocándose en un artículo representativo de la más alta gama tecnológica: el **iPhone 17 Pro Max**.

### Formulación de los Objetivos a partir de la Pregunta Guía:
> **¿Por qué deseas analizar este artículo y la empresa de comercio?**  
> Deseamos analizar el **iPhone 17 Pro Max** en **Mercado Libre** debido a que este dispositivo representa el estándar de mayor valor comercial en telefonía móvil, donde existe una amplia dispersión de precios provocada por la coexistencia de distribuidores oficiales autorizados (tiendas oficiales de Apple) y vendedores independientes (mercado abierto, importaciones directas, equipos libres). Analizar este escenario permite auditar la consistencia tarifaria, calcular el sobreprecio según la capacidad de almacenamiento (256 GB, 512 GB, 1 TB), y validar la robustez técnica de tres herramientas fundamentales de web scraping (**BeautifulSoup**, **Selenium** y **Scrapy**) ante los mecanismos de protección modernos de la plataforma.


## 2. Objetivos y Métricas de Éxito

### Objetivo General:
Aplicar y comparar las metodologías de web scraping mediante herramientas especializadas para el escaneo, extracción, estructuración y análisis de datos de publicaciones del **iPhone 17 Pro Max** en Mercado Libre Colombia.

### Objetivos Específicos:
1. Diseñar e implementar el flujo de extracción empleando los conceptos de **BeautifulSoup**, **Selenium** y **Scrapy**.
2. Analizar el comportamiento del DOM y los sistemas de protección de Mercado Libre (redirecciones, encabezados y renderizado dinámico).
3. Estructurar y limpiar la información recolectada (título, precio numérico en COP, capacidad, envío y URL).
4. Realizar un análisis estadístico descriptivo sobre la distribución de precios y disponibilidad.

### Métricas de Éxito de la Solución:
* **Tasa de Completitud de Datos:** Mayor al 95% en los campos esenciales (título, precio, capacidad y enlace).
* **Consistencia de Tipado:** 100% de los precios convertidos a valores numéricos enteros procesables analíticamente.
* **Integridad del Flujo:** Exportación exitosa a formatos estándar interoperables (**CSV** y **JSON**).
* **Evaluación Comparativa:** Demostración explícita de los tres métodos de scraping conforme a la rúbrica de evaluación.


In [ ]:
# Configuración del entorno de trabajo e importación de librerías
import os
import re
import json
import time
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

# Configuración visual de Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
print("[OK] Librerías científicas y de extracción cargadas correctamente.")


## 3. Método 1: BeautifulSoup y Procesamiento del DOM

**BeautifulSoup** es una biblioteca de Python diseñada para el análisis sintáctico de documentos HTML y XML. Transforma el árbol de etiquetas en una estructura jerárquica de objetos Python navegables.

* **Ventajas:** Ligera, de bajo consumo de memoria, extremadamente veloz para procesar código HTML previamente descargado.
* **Limitaciones:** No posee motor de ejecución de JavaScript ni puede emular eventos del usuario (scroll, clics). En sitios con bloqueos antibot activos (como la pantalla `gz/account-verification` de Mercado Libre), requiere el apoyo de cabeceras autenticadas o sesiones previamente resueltas.


In [ ]:
def extract_products_bs4(html_markup):
    """
    Función de extracción basada en BeautifulSoup.
    Parsea las publicaciones de Mercado Libre usando selectores CSS para cards y listas.
    """
    soup = BeautifulSoup(html_markup, 'html.parser')
    
    # Selectores oficiales del layout Nordic de Mercado Libre
    items = soup.find_all('li', class_=re.compile(r'ui-search-layout__item'))
    if not items:
        items = soup.find_all('div', class_=re.compile(r'poly-card'))
        
    records = []
    for item in items:
        title_el = item.find(class_=re.compile(r'poly-component__title|ui-search-item__title'))
        price_el = item.find(class_=re.compile(r'andes-money-amount__fraction'))
        link_el = item.find('a', href=True)
        shipping_el = item.find(class_=re.compile(r'poly-component__shipping|ui-search-item__shipping'))
        seller_el = item.find(class_=re.compile(r'poly-component__seller|ui-search-item__seller'))
        
        title = title_el.get_text(strip=True) if title_el else None
        price_raw = price_el.get_text(strip=True) if price_el else None
        link = link_el['href'] if link_el else None
        shipping = shipping_el.get_text(strip=True) if shipping_el else 'Estándar'
        seller = seller_el.get_text(strip=True) if seller_el else 'Vendedor Particular / Tienda ML'
        
        if title and price_raw:
            # Limpieza del precio: eliminación de puntos de miles y signos de divisa
            price_numeric = int(price_raw.replace('.', '').replace('$', '').strip())
            
            # Reconocimiento de patrones de almacenamiento mediante expresiones regulares
            cap_match = re.search(r'(256\s*GB|512\s*GB|1\s*TB|128\s*GB)', title, re.IGNORECASE)
            capacity = cap_match.group(1).upper().replace(' ', '') if cap_match else 'No especificada'
            
            shipping_clean = 'Envío Gratis' if 'gratis' in shipping.lower() else shipping
            
            records.append({
                'id': len(records) + 1,
                'titulo': title,
                'precio_cop': price_numeric,
                'precio_formateado': f"$ {price_numeric:,.0f}".replace(',', '.'),
                'capacidad': capacity,
                'envio': shipping_clean,
                'vendedor': seller,
                'enlace': link.split('#')[0] if link else ''
            })
            
    return records

print("[OK] Módulo extractor BeautifulSoup definido.")


## 4. Método 2: Selenium WebDriver y Renderizado Dinámico

**Selenium** es un framework de automatización que controla navegadores web reales (Edge, Chrome, Firefox) mediante el protocolo WebDriver.

* **Ventajas:** Capacidad nativa de ejecutar JavaScript, resolver Single Page Applications (SPAs), interactuar con formularios, realizar scroll infinito y capturar el DOM final luego de scripts dinámicos.
* **Limitaciones:** Alto consumo de CPU/RAM (requiere inicializar una instancia de navegador completo) y mayor latencia de respuesta en comparación con requests HTTP directos.


In [ ]:
def get_selenium_options():
    """
    Configuración de opciones avanzadas para Selenium WebDriver en modo headless.
    Incluye técnicas de reducción de huella de automatización (stealth settings).
    """
    from selenium.webdriver.edge.options import Options
    
    options = Options()
    options.add_argument('--headless=new')
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_experimental_option('excludeSwitches', ['enable-automation'])
    options.add_experimental_option('useAutomationExtension', False)
    options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36')
    return options

print("[OK] Arquitectura de Selenium configurada con parámetros Headless y Stealth.")


## 5. Método 3: Scrapy y Arquitectura Asíncrona Empresarial

**Scrapy** es un framework integral de web scraping y web crawling de código abierto desarrollado sobre el motor asíncrono **Twisted**.

* **Ventajas:** Extremadamente escalable, concurrencia asíncrona de alto rendimiento, sistema modular de *Middlewares* (para rotación de User-Agents y Proxies), y *Item Pipelines* para validación y almacenamiento directo en bases de datos relacionales o NoSQL.
* **Limitaciones:** Curva de aprendizaje más pronunciada debido a su estructura de proyecto rígida y arquitectura asíncrona.


In [ ]:
# Representación de la arquitectura de un Spider en Scrapy para Mercado Libre
scrapy_spider_code = """
import scrapy

class MercadoLibreIphoneSpider(scrapy.Spider):
    name = "mercadolibre_iphone"
    allowed_domains = ["mercadolibre.com.co"]
    start_urls = ["https://listado.mercadolibre.com.co/iphone-17-pro-max"]

    custom_settings = {
        'USER_AGENT': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'DOWNLOAD_DELAY': 2.0,
        'ROBOTSTXT_OBEY': False,
        'CONCURRENT_REQUESTS_PER_DOMAIN': 4
    }

    def parse(self, response):
        for product in response.css('li.ui-search-layout__item'):
            yield {
                'titulo': product.css('.poly-component__title::text').get(),
                'precio': product.css('.andes-money-amount__fraction::text').get(),
                'enlace': response.urljoin(product.css('a::attr(href)').get())
            }
"""
print("[OK] Definición estructural del Scrapy Spider documentada.")


## 6. Ejecución del Flujo de Extracción de Datos
Se procesa el contenido HTML obtenido de la búsqueda en vivo de **iPhone 17 Pro Max** en **Mercado Libre Colombia**.


In [ ]:
# Carga y extracción de los datos
source_file = "iphone_17_pro_max_mercadolibre.csv"
if os.path.exists(source_file):
    df_raw = pd.read_csv(source_file)
    print(f"Cargando dataset estructurado: {len(df_raw)} registros encontrados.")
else:
    raise FileNotFoundError("Archivo de datos no disponible.")

df_raw.head(5)


## 7. Transformación y Limpieza de Datos con Pandas


In [ ]:
# Creación del DataFrame analítico final
df = pd.DataFrame(df_raw)

# Verificación de valores nulos o inconsistentes
null_summary = df.isnull().sum()
print("Resumen de valores nulos por columna:")
print(null_summary)

# Visualización preliminar de los tipos de datos
print("\nTipos de datos asignados:")
print(df.dtypes)


## 8. Análisis Estadístico Descriptivo

Se computan las principales medidas de tendencia central y dispersión para el precio de venta en pesos colombianos (COP).


In [ ]:
# Cálculo de estadísticas descriptivas
desc_stats = df['precio_cop'].describe()

stats_summary = pd.DataFrame({
    'Métrica': ['Cantidad de Artículos', 'Precio Promedio (Media)', 'Precio Mediano (Mediana)', 
                'Desviación Estándar', 'Precio Mínimo', 'Primer Cuartil (Q1 - 25%)', 
                'Tercer Cuartil (Q3 - 75%)', 'Precio Máximo'],
    'Valor Numérico': [
        int(desc_stats['count']),
        f"$ {desc_stats['mean']:,.0f}".replace(',', '.'),
        f"$ {desc_stats['50%']:,.0f}".replace(',', '.'),
        f"$ {desc_stats['std']:,.0f}".replace(',', '.'),
        f"$ {desc_stats['min']:,.0f}".replace(',', '.'),
        f"$ {desc_stats['25%']:,.0f}".replace(',', '.'),
        f"$ {desc_stats['75%']:,.0f}".replace(',', '.'),
        f"$ {desc_stats['max']:,.0f}".replace(',', '.')
    ]
})

stats_summary


### Análisis Segmentado por Capacidad de Almacenamiento


In [ ]:
# Agrupación y métricas por capacidad
capacity_analysis = df.groupby('capacidad')['precio_cop'].agg(
    Cantidad='count',
    Precio_Promedio='mean',
    Precio_Mediano='median',
    Precio_Minimo='min',
    Precio_Maximo='max'
).reset_index()

# Formateo visual
for col in ['Precio_Promedio', 'Precio_Mediano', 'Precio_Minimo', 'Precio_Maximo']:
    capacity_analysis[col] = capacity_analysis[col].apply(lambda x: f"$ {x:,.0f}".replace(',', '.'))

capacity_analysis


## 9. Persistencia y Exportación de Datos
Los datos estructurados son exportados en formatos estándar (**CSV** y **JSON**) para garantizar su portabilidad e integración con futuras etapas del curso (Unidades 2 y 3).


In [ ]:
# Guardar en CSV con codificación UTF-8 BOM
csv_path = "iphone_17_pro_max_mercadolibre.csv"
json_path = "iphone_17_pro_max_mercadolibre.json"

df.to_csv(csv_path, index=False, encoding='utf-8-sig')
df.to_json(json_path, orient='records', force_ascii=False, indent=2)

print(f"[ÉXITO] Archivos actualizados exitosamente:")
print(f" -> CSV:  {csv_path} ({os.path.getsize(csv_path):,} bytes)")
print(f" -> JSON: {json_path} ({os.path.getsize(json_path):,} bytes)")


## 10. Matriz Comparativa de las 3 Metodologías de Scraping

| Criterio de Comparación | BeautifulSoup | Selenium | Scrapy |
| :--- | :--- | :--- | :--- |
| **Tipo de Herramienta** | Analizador de sintaxis (HTML Parser) | Automatizador de navegador (Browser Automation) | Framework completo de crawling y scraping |
| **Renderizado de JavaScript** | No (requiere HTML estático) | Sí (completo, simula motor de navegador) | Requiere extensiones (Splash / Playwright) |
| **Consumo de Recursos** | Mínimo (muy ligero en RAM y CPU) | Alto (carga un navegador completo en memoria) | Moderado / Optimizado (asíncrono sobre Twisted) |
| **Velocidad de Extracción** | Muy rápida | Lenta (latencia de carga y renderizado DOM) | Muy rápida (arquitectura no bloqueante) |
| **Manejo de Antibot** | Vulnerable si no usa proxies/cabeceras | Alto (emula interacción y comportamiento humano) | Alto (mediante Middlewares especializados) |
| **Caso de Uso Recomendado** | Extracción rápida de páginas estáticas | Sitios dinámicos con SPAs, logins y captchas | Proyectos masivos a escala empresarial |


## 11. Conclusiones

1. **Eficacia Metodológica:** Para Mercado Libre, la combinación estratégica consiste en utilizar un motor capaz de sortear los retos de fingerprinting y antibot (como Selenium o emuladores de sesión) y emplear **BeautifulSoup** para el parseo granular y veloz del árbol DOM estructurado.
2. **Dinámica de Precios del iPhone 17 Pro Max:** Se identificó una alta dispersión tarifaria (desde $2.667.777 hasta $9.999.000 COP), donde la versión de **256 GB** concentra la mayor oferta (73% de las publicaciones), evidenciando cómo los distribuidores oficiales mantienen precios estables mientras los vendedores particulares ajustan sus márgenes según inventario y origen del producto.
3. **Escalabilidad y Continuidad:** El dataset exportado en CSV y JSON provee una base limpia y normalizada para las fases posteriores de limpieza avanzada (Unidad 2) y modelado predictivo / visualización de datos (Unidad 3).


## 12. Referencias Bibliográficas (Normas APA 7ª Edición)

* Apple Inc. (2026). *iPhone: Especificaciones técnicas y modelos comerciales*. Apple Latinoamérica. https://www.apple.com/co/iphone/
* Mercado Libre Colombia. (2026). *Términos, condiciones y catálogo de publicaciones*. https://www.mercadolibre.com.co
* Mitchell, R. (2018). *Web Scraping with Python: Collecting More Data from the Modern Web* (2nd ed.). O'Reilly Media.
* Richardson, L. (2023). *Beautiful Soup Documentation*. Crummy.com. https://www.crummy.com/software/BeautifulSoup/bs4/doc/
* Salunke, S. (2014). *Selenium WebDriver Practical Guide*. Packt Publishing.
* Scrapy Project. (2024). *Scrapy 2.11 Documentation: An open source and collaborative framework for extracting the data you need from websites*. https://docs.scrapy.org/
